# Strands Agent with Langfuse Observability on Amazon Bedrock AgentCore Runtime

## Overview

This notebook demonstrates deploying a Strands agent to Amazon Bedrock AgentCore Runtime with Langfuse observability integration. The implementation uses Amazon Bedrock Claude models and sends telemetry data to Langfuse through OpenTelemetry (OTEL).

## Key Components

- **Strands Agents**: Python framework for building LLM-powered agents with built-in telemetry support
- **Amazon Bedrock AgentCore Runtime**: Managed runtime service for hosting and scaling agents on AWS
- **Langfuse**: Open-source observability platform for LLM applications that receives traces via OTEL
- **OpenTelemetry**: Industry-standard protocol for collecting and exporting telemetry data

## Architecture

The agent is containerized and deployed to AgentCore Runtime, which provides HTTP endpoints for invocation. Telemetry data flows from the Strands agent through OTEL exporters to Langfuse for monitoring and debugging. The implementation disables AgentCore's default observability to use Langfuse instead.

## Prerequisites

- Python 3.10+
- AWS credentials configured with Bedrock and AgentCore permissions
- [Langfuse](https://langfuse.com/) account with API keys (public and secret keys)
- Docker installed locally
- Access to Amazon Bedrock Claude models in us-west-2

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Configure AWS Credentials

## Agent Implementation

The agent file (`strands_claude.py`) implements a travel agent with web search capabilities. Key configuration includes:
- Disabling AgentCore's default OTEL configuration
- Setting Langfuse endpoint and authentication
- Initializing Strands telemetry after environment variables are configured

In [ ]:
%%writefile strands_claude.py
import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from ddgs import DDGS

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "ERROR").upper())


@tool
def web_search(query: str) -> str:
    """
    Search the web for information using DuckDuckGo.

    Args:
        query: The search query

    Returns:
        A string containing the search results
    """
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )

        return "\n".join(formatted_results) if formatted_results else "No results found."

    except Exception as e:
        return f"Error searching the web: {str(e)}"

# Function to initialize Bedrock model
def get_bedrock_model():
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")
    model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-3-7-sonnet-20250219-v1:0")

    bedrock_model = BedrockModel(
        model_id=model_id,
        region_name=region,
        temperature=0.0,
        max_tokens=1024
    )
    return bedrock_model

# Initialize the Bedrock model
bedrock_model = get_bedrock_model()

# Define the agent's system prompt
system_prompt = """You are an experienced travel agent specializing in personalized travel recommendations 
with access to real-time web information. Your role is to find dream destinations matching user preferences 
using web search for current information. You should provide comprehensive recommendations with current 
information, brief descriptions, and practical travel details."""

app = BedrockAgentCoreApp()

def initialize_agent():
    """Initialize the agent with proper telemetry configuration."""

    # Initialize Strands telemetry with 3P configuration
    strands_telemetry = StrandsTelemetry()
    strands_telemetry.setup_otlp_exporter()
    
    # Create and cache the agent
    agent = Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[web_search]
    )
    
    return agent

@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    logger.info("[%s] User input: %s", context.session_id, user_input)
    
    # Initialize agent with proper configuration
    agent = initialize_agent()
    
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

Overwriting strands_claude.py


### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it configures AgentCore Observability by default so, to use Braintrust, you need to remove configuration for AgentCore Observability as explained below:

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [3]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_langfuse_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    disable_otel=True,
)
response

Entrypoint parsed: file=/Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/strands_claude.py, bedrock_agentcore_name=strands_claude
Configuring BedrockAgentCore agent: strands_langfuse_observability
Generated .dockerignore
Generated Dockerfile: /Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/Dockerfile
Generated .dockerignore: /Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/.dockerignore
Setting 'strands_langfuse_observability' as default agent
Bedrock AgentCore configured: /Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/

ConfigureResult(config_path=PosixPath('/Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/Dockerfile'), dockerignore_path=PosixPath('/Users/ravizraj/Documents/workspaces/agent-core/rajesh/amazon-bedrock-agentcore-samples/01-tutorials/06-AgentCore-observability/04-Agentcore-runtime-partner-observability/Langfuse/.dockerignore'), runtime='Finch', region='us-east-1', account_id='467801433859', execution_role=None, ecr_repository=None, auto_create_ecr=True)

## Deploy to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [30]:
import base64

# Langfuse configuration
otel_endpoint = "https://us.cloud.langfuse.com/api/public/otel"
langfuse_secret_key = "sk-lf-d5d2be8d-5705-4474-8b52-c60d03cfbec3"
langfuse_public_key = "pk-lf-294eef9b-4f1b-4287-8f31-fc0d444a070f"
langfuse_auth_token = base64.b64encode(f"{langfuse_public_key}:{langfuse_secret_key}".encode()).decode()
otel_auth_header = f"Authorization=Basic {langfuse_auth_token}"


launch_result = agentcore_runtime.launch(
    env_vars={
        "BEDROCK_MODEL_ID": "us.anthropic.claude-3-7-sonnet-20250219-v1:0", # Example model ID
        "OTEL_EXPORTER_OTLP_ENDPOINT": otel_endpoint,  # Use Langfuse OTEL endpoint
        "OTEL_EXPORTER_OTLP_HEADERS": otel_auth_header,  # Add Langfuse OTEL auth header
        "DISABLE_ADOT_OBSERVABILITY": "true",
        "AGENT_RUNTIME_LOG_LEVEL": "info", # optional custom logs from agent runtime [default=error]
    }
)
launch_result


🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'strands_langfuse_observability' to account 467801433859 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Using ECR repository from config: 467801433859.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-strands_langfuse_observability
Using execution role from config: arn:aws:iam::467801433859:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-2102f51ebb
Preparing CodeBuild project and uploading source...
Using CodeBuild role from config: arn:aws:iam::467801433859:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-2102f51ebb
Using 

LaunchResult(mode='codebuild', tag='bedrock_agentcore-strands_langfuse_observability:latest', env_vars=None, port=None, runtime=None, ecr_uri='467801433859.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-strands_langfuse_observability', agent_id='strands_langfuse_observability-64dAhNEayV', agent_arn='arn:aws:bedrock-agentcore:us-east-1:467801433859:runtime/strands_langfuse_observability-64dAhNEayV', codebuild_id='bedrock-agentcore-strands_langfuse_observability-builder:36f1a496-5d77-4542-9d99-93a7288f72b6', build_output=None)

## Check Deployment Status

Wait for the runtime to be ready before invoking:

In [15]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

Retrieved Bedrock AgentCore status for: strands_langfuse_observability


'READY'

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="../images/invoke.png" width=75%"/>
</div>

In [31]:
from IPython.display import Markdown, display

invoke_response = agentcore_runtime.invoke({"prompt": "I'm planning a weekend trip to munich. What are the must-visit places and local food I should try?"})

display(Markdown("".join(invoke_response['response'])))

"# Munich Weekend Trip: Must-Visit Places and Local Food\n\nBased on the latest information, here's a comprehensive guide for your weekend trip to Munich:\n\n## Must-Visit Places in Munich\n\n1. **Marienplatz** - The central square of Munich featuring the New Town Hall (Neues Rathaus) with its famous Glockenspiel clock show that performs daily at 11am, 12pm, and 5pm (summer only).\n\n2. **English Garden (Englischer Garten)** - One of the world's largest urban parks, larger than Central Park in New York. Visit the Chinese Tower beer garden, watch surfers on the artificial wave at Eisbach, or simply enjoy a relaxing stroll.\n\n3. **Nymphenburg Palace** - A magnificent baroque palace with beautiful gardens and museums, once the summer residence of Bavarian rulers.\n\n4. **BMW Museum and BMW World** - Perfect for car enthusiasts, showcasing the history and future of BMW vehicles.\n\n5. **Viktualienmarkt** - A daily food market and square in the center of Munich, offering fresh produce, flowers, and specialty foods.\n\n6. **Munich Residenz** - The former royal palace of the Bavarian monarchs, featuring impressive architecture, museums, and the Treasury.\n\n7. **Deutsches Museum** - One of the world's largest science and technology museums.\n\n8. **Frauenkirche (Cathedral of Our Lady)** - Munich's iconic cathedral with its distinctive twin towers that dominate the city skyline.\n\n9. **Olympiapark** - Built for the 1972 Summer Olympics, offering great views from the Olympic Tower and beautiful grounds.\n\n10. **Hofbräuhaus** - Munich's most famous beer hall, dating back to 1589.\n\n## Traditional Bavarian Food to Try\n\n1. **Weisswurst** - Traditional Bavarian white sausage typically eaten for breakfast with sweet mustard, pretzels, and wheat beer.\n\n2. **Schweinshaxe** - Crispy roasted pork knuckle, a hearty Bavarian specialty.\n\n3. **Leberkäse** - A meatloaf-like dish often served on a roll as a quick meal.\n\n4. **Bretzel (Pretzel)** - Soft, doughy pretzels, often enjoyed with beer or as a snack.\n\n5. **Obatzda** - A savory cheese spread made with camembert, butter, and paprika, typically served with pretzels or bread.\n\n6. **Käsespätzle** - The Bavarian version of mac and cheese, made with fresh egg noodles and topped with crispy fried onions.\n\n7. **Apfelstrudel** - A warm apple pastry dessert, often served with vanilla sauce or ice cream.\n\n8. **Kaiserschmarrn** - Shredded pancake dessert served with applesauce or fruit compote.\n\n9. **Bavarian Beer** - Munich is famous for its beer culture. Try local varieties like Helles (light lager), Weissbier (wheat beer), or Dunkel (dark beer).\n\n10. **Dampfnudel** - Sweet steamed dumplings served with vanilla sauce or compote.\n\n## Practical Tips:\n\n- **Getting Around**: Munich has an excellent public transportation system (MVV) with U-Bahn (subway), S-Bahn (suburban trains), trams, and buses. Consider getting a day pass for unlimited travel.\n\n- **Best Time to Visit**: Weekends can be busy at major attractions. Try to visit popular sites early in the morning.\n\n- **Traditional Restaurants**: For authentic Bavarian cuisine, visit traditional \"Wirtshäuser\" like Hofbräuhaus, Augustiner Bräustuben, or Schneider Bräuhaus.\n\n- **Beer Gardens**: If weather permits, experience Munich's beer garden culture at places like Augustiner-Keller, Chinesischer Turm (in the English Garden), or Viktualienmarkt.\n\nWould you like more specific information about any of these attractions or food recommendations? Or perhaps details about transportation or accommodations in Munich?"

## View Traces in Langfuse

To view the traces:
1. Go to your Langfuse dashboard at https://cloud.langfuse.com
2. Navigate to your project
3. Click on "Traces" to view the telemetry data

The traces will include:
- Agent invocation details
- Tool calls (web search)
- Model interactions with latency and token usage
- Request/response payloads

## Cleanup (Optional)

Clean up the deployed resources:

In [ ]:
import boto3

agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)

ecr_client = boto3.client(
    'ecr',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

## Summary

You have successfully deployed a Strands agent to Amazon Bedrock AgentCore Runtime with Langfuse observability. The implementation demonstrates:
- Integration of Strands agents with AgentCore Runtime
- Configuration of OpenTelemetry to send traces to Langfuse
- Proper initialization order to ensure telemetry configuration
- Invocation through both SDK and boto3 client

The agent is now running in a managed, scalable environment with full observability through Langfuse.